In [3]:
from pathlib import Path
import sys
from datetime import date
from pyspark.sql import functions as F

# On remonte les dossiers jusqu'à trouver la racine du projet (celle qui contient "ingestion")
BASE = Path.cwd()
while not (BASE / "ingestion").exists() and BASE != BASE.parent:
    BASE = BASE.parent

sys.path.append(str(BASE))
from ingestion.spark_session import get_spark

spark = get_spark("explore")

RAW = BASE / "data" / "raw"
BRONZE = BASE / "data" / "bronze"

print("Racine du projet :", BASE)
print("Dossier raw existe :", RAW.exists())
print("CSV trouvés :", [p.name for p in RAW.glob("*.csv")])

Racine du projet : C:\Users\User\ecommerce-data-pipeline
Dossier raw existe : True
CSV trouvés : ['olist_customers_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_products_dataset.csv']


In [6]:
orders = (
    spark.read
    .option("header", True)
    .csv(str(RAW / "olist_orders_dataset.csv"))
)

orders.printSchema()
orders.show(5, truncate=False)
print(orders.count(), "lignes,", len(orders.columns), "colonnes")

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+-------------------------

In [7]:
# order_id est-il une clé unique ?
orders.agg(
    F.count("*").alias("lignes"),
    F.countDistinct("order_id").alias("order_id_distincts")
).show()

#  Combien de dates de livraison manquent ?
orders.select(
    F.sum(F.col("order_delivered_customer_date").isNull().cast("int")).alias("livraison_manquante")
).show()

#  Répartition par statut
orders.groupBy("order_status").count().orderBy(F.desc("count")).show()

+------+------------------+
|lignes|order_id_distincts|
+------+------------------+
| 99441|             99441|
+------+------------------+

+-------------------+
|livraison_manquante|
+-------------------+
|               2965|
+-------------------+

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [8]:
orders_check = orders.withColumn(
    "livraison_manquante",
    F.col("order_delivered_customer_date").isNull()
)

(
    orders_check
    .groupBy("order_status", "livraison_manquante")
    .count()
    .orderBy("order_status", "livraison_manquante")
    .show(20)
)

+------------+-------------------+-----+
|order_status|livraison_manquante|count|
+------------+-------------------+-----+
|    approved|               true|    2|
|    canceled|              false|    6|
|    canceled|               true|  619|
|     created|               true|    5|
|   delivered|              false|96470|
|   delivered|               true|    8|
|    invoiced|               true|  314|
|  processing|               true|  301|
|     shipped|               true| 1107|
| unavailable|               true|  609|
+------------+-------------------+-----+



In [9]:
anomalies = orders.filter(
    (F.col("order_status") == "delivered") &
    F.col("order_delivered_customer_date").isNull()
)

print("Commandes 'delivered' sans date de livraison :", anomalies.count())
anomalies.select(
    "order_id", "order_status",
    "order_delivered_carrier_date", "order_delivered_customer_date"
).show(10, truncate=False)

Commandes 'delivered' sans date de livraison : 8
+--------------------------------+------------+----------------------------+-----------------------------+
|order_id                        |order_status|order_delivered_carrier_date|order_delivered_customer_date|
+--------------------------------+------------+----------------------------+-----------------------------+
|2d1e2d5bf4dc7227b3bfebb81328c15f|delivered   |2017-11-30 18:12:23         |null                         |
|f5dd62b788049ad9fc0526e3ad11a097|delivered   |2018-06-25 08:05:00         |null                         |
|2ebdfc4f15f23b91474edf87475f108e|delivered   |2018-07-03 13:57:00         |null                         |
|e69f75a717d64fc5ecdfae42b2e8e086|delivered   |2018-07-03 13:57:00         |null                         |
|0d3268bad9b086af767785e3f0fc0133|delivered   |2018-07-03 09:28:00         |null                         |
|2d858f451373b04fb5c984a1cc2defaf|delivered   |null                        |null               

In [10]:
%%time
orders_inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "olist_orders_dataset.csv"))
)
orders_inferred.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

CPU times: total: 0 ns
Wall time: 1.76 s


In [11]:
c_raw = spark.read.option("header", True).csv(str(RAW / "olist_customers_dataset.csv"))
c_inf = spark.read.option("header", True).option("inferSchema", True).csv(str(RAW / "olist_customers_dataset.csv"))

c_raw.printSchema()
c_inf.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [12]:
run_date = date.today().isoformat()

orders_bronze = (
    orders
    .withColumn("_ingestion_date", F.lit(run_date))
    .withColumn("_source_file", F.lit("olist_orders_dataset.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
)

orders_bronze.select("order_id", "_ingestion_date", "_source_file", "_ingested_at").show(3, truncate=False)

(
    orders_bronze.write
    .mode("overwrite")
    .partitionBy("_ingestion_date")
    .parquet(str(BRONZE / "orders"))
)
print("Bronze orders écrit")

+--------------------------------+---------------+------------------------+--------------------------+
|order_id                        |_ingestion_date|_source_file            |_ingested_at              |
+--------------------------------+---------------+------------------------+--------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|2026-10-18     |olist_orders_dataset.csv|2026-10-18 15:37:29.715786|
|53cdb2fc8bc7dce0b6741e2150273451|2026-10-18     |olist_orders_dataset.csv|2026-10-18 15:37:29.715786|
|47770eb9100c2d0c44946d9cf07ec65d|2026-10-18     |olist_orders_dataset.csv|2026-10-18 15:37:29.715786|
+--------------------------------+---------------+------------------------+--------------------------+
only showing top 3 rows

Bronze orders écrit


In [13]:
check = spark.read.parquet(str(BRONZE / "orders"))
print("Lignes relues :", check.count())
check.select("_ingestion_date").distinct().show()
print("Dossiers créés :", [p.name for p in (BRONZE / "orders").iterdir()])

Lignes relues : 99441
+---------------+
|_ingestion_date|
+---------------+
|     2026-10-18|
+---------------+

Dossiers créés : ['_ingestion_date=2026-10-18']


In [14]:
items = (
    spark.read.option("header", True)
    .csv(str(RAW / "olist_order_items_dataset.csv"))
)
items.printSchema()
items.show(5, truncate=False)

items.agg(
    F.count("*").alias("lignes"),
    F.countDistinct("order_id").alias("order_id_distincts"),
    F.countDistinct("order_id", "order_item_id").alias("paires_distinctes")
).show()

# Combien de commandes ont plus d'un article ?
(
    items.groupBy("order_id").count()
    .filter(F.col("count") > 1)
    .count()
)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.90 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee65886

9803

In [15]:
SOURCES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
}

def ingest_table(name, filename, run_date):
    src = RAW / filename
    if not src.exists():
        raise FileNotFoundError(f"Fichier introuvable : {src}")

    df = spark.read.option("header", True).csv(str(src))
    df = (
        df.withColumn("_ingestion_date", F.lit(run_date))
          .withColumn("_source_file", F.lit(filename))
          .withColumn("_ingested_at", F.current_timestamp())
    )
    n = df.count()
    df.write.mode("overwrite").partitionBy("_ingestion_date").parquet(str(BRONZE / name))
    print(f"OK {name:12s} | {n:8d} lignes")

for name, filename in SOURCES.items():
    ingest_table(name, filename, run_date)

OK orders       |    99441 lignes
OK order_items  |   112650 lignes
OK products     |    32951 lignes
OK customers    |    99441 lignes
OK payments     |   103886 lignes


In [16]:
for name, filename in SOURCES.items():
    n_csv = spark.read.option("header", True).csv(str(RAW / filename)).count()
    n_bronze = spark.read.parquet(str(BRONZE / name)).count()
    statut = "OK" if n_csv == n_bronze else "ÉCART !"
    print(f"{name:12s} csv={n_csv:8d} | bronze={n_bronze:8d} | {statut}")

orders       csv=   99441 | bronze=   99441 | OK
order_items  csv=  112650 | bronze=  112650 | OK
products     csv=   32951 | bronze=   32951 | OK
customers    csv=   99441 | bronze=   99441 | OK
payments     csv=  103886 | bronze=  103886 | OK


In [17]:
orders_b = spark.read.parquet(str(BRONZE / "orders"))
items_b = spark.read.parquet(str(BRONZE / "order_items"))

# Commandes qui n'ont AUCUN article
sans_items = orders_b.join(items_b, "order_id", "left_anti")
print("Commandes sans article :", sans_items.count())
sans_items.groupBy("order_status").count().orderBy(F.desc("count")).show()

# Articles dont la commande n'existe pas
items_orphelins = items_b.join(orders_b, "order_id", "left_anti")
print("Articles orphelins :", items_orphelins.count())

Commandes sans article : 775
+------------+-----+
|order_status|count|
+------------+-----+
| unavailable|  603|
|    canceled|  164|
|     created|    5|
|    invoiced|    2|
|     shipped|    1|
+------------+-----+

Articles orphelins : 0


In [18]:
def nulls_par_colonne(df):
    return df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
    ])

for name in SOURCES:
    print("==", name)
    nulls_par_colonne(spark.read.parquet(str(BRONZE / name))).show(truncate=False)

== orders
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+------------+---------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_source_file|_ingested_at|_ingestion_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+------------+---------------+
|0       |0          |0           |0                       |160              |1783                        |2965                         |0                            |0           |0           |0              |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-------------

In [19]:
c_raw.filter(F.col("customer_zip_code_prefix").startswith("0")) \
     .select("customer_zip_code_prefix").show(3)

c_inf.filter(F.col("customer_zip_code_prefix") < 10000) \
     .select("customer_zip_code_prefix").show(3)

+------------------------+
|customer_zip_code_prefix|
+------------------------+
|                   09790|
|                   01151|
|                   08775|
+------------------------+
only showing top 3 rows

+------------------------+
|customer_zip_code_prefix|
+------------------------+
|                    9790|
|                    1151|
|                    8775|
+------------------------+
only showing top 3 rows



In [20]:
products_b = spark.read.parquet(str(BRONZE / "products"))

sans_categorie = products_b.filter(F.col("product_category_name").isNull())
sans_categorie.select(
    F.count("*").alias("lignes"),
    F.sum(F.col("product_name_lenght").isNull().cast("int")).alias("nom_null"),
    F.sum(F.col("product_description_lenght").isNull().cast("int")).alias("desc_null"),
    F.sum(F.col("product_photos_qty").isNull().cast("int")).alias("photos_null"),
).show()

+------+--------+---------+-----------+
|lignes|nom_null|desc_null|photos_null|
+------+--------+---------+-----------+
|   610|     610|      610|        610|
+------+--------+---------+-----------+



In [21]:
items_b = spark.read.parquet(str(BRONZE / "order_items"))
payments_b = spark.read.parquet(str(BRONZE / "payments"))

items_b.select(
    F.min(F.col("price").cast("double")).alias("prix_min"),
    F.max(F.col("price").cast("double")).alias("prix_max"),
    F.min(F.col("freight_value").cast("double")).alias("frais_min"),
    F.max(F.col("freight_value").cast("double")).alias("frais_max"),
).show()

payments_b.select(
    F.min(F.col("payment_value").cast("double")).alias("paiement_min"),
    F.max(F.col("payment_value").cast("double")).alias("paiement_max"),
).show()

+--------+--------+---------+---------+
|prix_min|prix_max|frais_min|frais_max|
+--------+--------+---------+---------+
|    0.85|  6735.0|      0.0|   409.68|
+--------+--------+---------+---------+

+------------+------------+
|paiement_min|paiement_max|
+------------+------------+
|         0.0|    13664.08|
+------------+------------+



In [22]:
customers_b = spark.read.parquet(str(BRONZE / "customers"))
customers_b.agg(
    F.count("*").alias("lignes"),
    F.countDistinct("customer_id").alias("customer_id_distincts"),
    F.countDistinct("customer_unique_id").alias("clients_uniques"),
).show()

+------+---------------------+---------------+
|lignes|customer_id_distincts|clients_uniques|
+------+---------------------+---------------+
| 99441|                99441|          96096|
+------+---------------------+---------------+

